# 🏗️ Rufus — House Building AI Video Generator

**Fully AI-generated construction videos. No stock footage.**

Each stage of the build is generated by Wan2.1 / CogVideoX from text prompts.

---
### Before running:
1. `Runtime → Change runtime type → GPU → L4` ← **important**
2. Run cells top to bottom
3. First run takes ~20 minutes (model download). After that: ~10 minutes per video.


In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Check GPU & VRAM
# ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f"\nGPU: {gpu}")
print(f"VRAM: {vram:.1f} GB")

if vram >= 20:
    MODEL_BACKEND = 'wan2'      # Wan2.1 — best quality
    print("✅ Using Wan2.1 (best quality)")
elif vram >= 14:
    MODEL_BACKEND = 'cogvideo'  # CogVideoX-2B
    print("✅ Using CogVideoX-2B")
else:
    MODEL_BACKEND = None
    print("❌ Not enough VRAM — switch to L4 GPU")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Mount Google Drive
# ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/Rufus/construction'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/clips', exist_ok=True)
print(f"✅ Videos → {DRIVE_OUTPUT}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Install everything
# ─────────────────────────────────────────────────────────
print("Installing FFmpeg...")
!apt-get install -y ffmpeg > /dev/null 2>&1

print("Installing Ollama...")
!curl -fsSL https://ollama.ai/install.sh | sh > /dev/null 2>&1

print("Installing Python packages...")
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg
!pip install -q torch torchvision
!pip install -q httpx pyyaml rich click python-dotenv
!pip install -q kokoro-onnx soundfile faster-whisper yt-dlp
!pip install -q Pillow opencv-python-headless scikit-learn numpy
!pip install -q faiss-cpu qdrant-client pydantic pytrends
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("\n✅ All installed")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Clone Rufus
# ─────────────────────────────────────────────────────────
RUFUS_DIR = '/content/Rufus'
if os.path.exists(RUFUS_DIR):
    !cd {RUFUS_DIR} && git pull origin claude/yt-viral-automation-xc6h3
else:
    !git clone https://github.com/MisterRufus-code/Rufus.git {RUFUS_DIR}
    !cd {RUFUS_DIR} && git checkout claude/yt-viral-automation-xc6h3

os.chdir(RUFUS_DIR)
print(f"✅ Rufus ready at {RUFUS_DIR}")
!git log --oneline -3

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — API Keys
# ─────────────────────────────────────────────────────────
os.environ['PEXELS_API_KEY']   = 'PASTE_YOUR_PEXELS_KEY_HERE'
os.environ['OUTPUT_PATH']      = DRIVE_OUTPUT
os.environ['MEDIA_LIBRARY_PATH'] = f'{DRIVE_OUTPUT}/media_library'
os.environ['QDRANT_HOST']      = ''

# Write .env
from pathlib import Path
Path('.env').write_text(f"""PEXELS_API_KEY={os.environ['PEXELS_API_KEY']}
OUTPUT_PATH={DRIVE_OUTPUT}
MEDIA_LIBRARY_PATH={DRIVE_OUTPUT}/media_library
QDRANT_HOST=
""")
print("✅ Keys set")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Start Ollama + pull model
# ─────────────────────────────────────────────────────────
import subprocess, time, httpx

LLM_MODEL = 'llama3.1'  # change to gemma3:27b for L4, llama3.1:70b for A100

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
!ollama pull {LLM_MODEL}

# Update niche config to use this model
import yaml
p = Path('config/niches/construction.yaml')
d = yaml.safe_load(p.read_text())
d['model'] = LLM_MODEL
p.write_text(yaml.dump(d, allow_unicode=True, default_flow_style=False))
print(f"✅ Ollama running with {LLM_MODEL}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Generate AI video clips from construction prompts
# This is the CORE of the AI generation — runs before the pipeline
# Takes ~3-5 minutes per clip on L4
# ─────────────────────────────────────────────────────────
import sys
sys.path.insert(0, RUFUS_DIR)

from src.media_fetch.text_to_video import generate_scene_clips, is_available

# Construction scene prompts — one per build stage
SCENE_PROMPTS = [
    "cinematic drone shot of empty building plot, excavator breaking ground, workers measuring foundations, golden hour lighting, photorealistic 4K",
    "concrete being poured into foundation formwork, rebar reinforcement visible, construction workers in hard hats, cinematic close-up, photorealistic",
    "brick walls being laid course by course, masonry work, scaffolding rising, cinematic time-lapse, construction documentary, photorealistic 4K",
    "wooden roof trusses being lifted by crane onto house walls, workers installing rafters, wide angle cinematic shot, blue sky, photorealistic",
    "electrician running cables through wall studs, conduit installation, interior construction site, professional lighting, close-up detail, photorealistic",
    "plasterer applying smooth render to interior walls, skim coat, cinematic close-up, professional documentary style, photorealistic 4K",
    "finished house exterior with render and landscaping, front garden, cinematic wide shot, golden hour, photorealistic architecture photography",
]

print(f"Backend: {MODEL_BACKEND}")
print(f"Generating {len(SCENE_PROMPTS)} clips...")
print("This will take approximately", len(SCENE_PROMPTS) * 4, "minutes on L4\n")

if is_available():
    clips = generate_scene_clips(SCENE_PROMPTS, duration_each=5)
    print(f"\n✅ Generated {len(clips)} clips")
    for c in clips:
        print(f"   {c}")
else:
    print("❌ No GPU backend available — switch to L4 or A100")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Preview generated clips
# ─────────────────────────────────────────────────────────
from IPython.display import Video, display
import glob

ai_clips = sorted(glob.glob(f'{DRIVE_OUTPUT}/media_library/ai_generated/*.mp4'))
print(f"Found {len(ai_clips)} AI-generated clips")

# Show first clip
if ai_clips:
    print(f"\nPreviewing: {ai_clips[0]}")
    display(Video(ai_clips[0], embed=True, width=640))

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — 🚀 Run full Rufus pipeline with AI footage
# Script + voiceover + subtitles assembled automatically
# ─────────────────────────────────────────────────────────
os.chdir(RUFUS_DIR)
!python main.py pipeline --niche construction

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 10 — Watch the finished video
# ─────────────────────────────────────────────────────────
from IPython.display import Video, display
import glob

videos = sorted(glob.glob(f'{DRIVE_OUTPUT}/**/final_video.mp4', recursive=True))
if not videos:
    videos = sorted(glob.glob(f'{DRIVE_OUTPUT}/**/*.mp4', recursive=True))

if videos:
    latest = videos[-1]
    print(f"✅ {latest}")
    display(Video(latest, embed=True, width=900))
else:
    print("No video found — check pipeline output above")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 11 — Generate more AI clips with custom prompts
# Run this cell with any prompts you want
# ─────────────────────────────────────────────────────────
from src.media_fetch.text_to_video import generate_clip

# ← Edit these prompts
CUSTOM_PROMPTS = [
    "professional tiler laying large format marble tiles in modern bathroom, overhead shot, photorealistic 4K",
    "modern kitchen being fitted, white cabinets, quartz worktop installation, cinematic interior, photorealistic",
    "aerial drone flying over completed house with garden, cinematic pull-back shot, golden hour, photorealistic",
]

for prompt in CUSTOM_PROMPTS:
    clip = generate_clip(prompt, duration=5)
    if clip:
        display(Video(str(clip), embed=True, width=640))

---
## 💡 Tips

**Better prompts = better video:**
- Add: `cinematic`, `photorealistic`, `4K`, `professional photography`
- Add lighting: `golden hour`, `blue sky`, `warm interior lighting`
- Be specific: `excavator breaking clay soil` not just `construction`

**Clip cache:** Already-generated clips are cached by prompt hash.  
Run the same prompt again → instant (no re-generation).

**More videos:** Just run Cell 9 again — different topic each time.

**Session expired?** Run cells: 2, 4, 6, 7, 9 (skip install)
